In [0]:
import importlib
import sys
from pathlib import Path
from pyspark.sql import functions as F
project_root = str(Path.cwd().resolve().parent)
if project_root not in sys.path:
    sys.path.append(project_root)

import utils.storage_config as storage_config
from pyspark.sql import functions as F
importlib.reload(storage_config)
storage_config.spark = spark
storage_config.configure_storage()

In [0]:

df = spark.read.format("delta")\
    .load("abfss://bronze@secondstorage89.dfs.core.windows.net/order-items/")

In [0]:
df.show(10)

```sql
order_id = O001 → one order
order_item_id = 1 → first item in that order
order_item_id = 2 → second item
order_item_id = 3 → third item
```

freight value - shipping charges


In [0]:
df.printSchema()

In [0]:


df.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df.columns
]).show()

```md
order_id    order_item_id
A           1
A           2
A           3
```

In [0]:
df.groupBy("order_id") \
    .count() \
    .filter(F.col("count") > 1) \
    .show(40)

checking for duplicate records

In [0]:
df.groupBy("order_id", "order_item_id") \
    .count() \
    .filter(F.col("count") > 1) \
    .show()

check if price and freights values are null or not---

In [0]:
df.select(
    "price",
    "freight_value"
).describe().show()

fixed-precision decimals provide consistent two-decimal representation and avoid floating-point precision issues.

In [0]:
silver_order_items = (
    df
    .withColumn(
        "price",
        F.col("price").cast("decimal(12,2)")
    )
    .withColumn(
        "freight_value",
        F.col("freight_value").cast("decimal(12,2)")
    )
)

In [0]:
silver_order_items.printSchema()

In [0]:
silver_order_items.write \
    .format("delta") \
    .mode("overwrite") \
    .save(
        "abfss://silver@secondstorage89.dfs.core.windows.net/order-items/"
    )